In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ['GEMINI_API_KEY']:
    print("GEMINI_API_KEY is set.")
else:   
     print("GEMINI_API_KEY is not set.")

GEMINI_API_KEY is set.


In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Switching to 2.5 Flash-Lite for better free-tier reliability
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite", 
    temperature=0
)

print("Connected to Gemini 2.5 Flash-Lite!")

d:\Ai Data Engineering\RAG_POC\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


Connected to Gemini 2.5 Flash-Lite!


In [3]:
try:
	response = llm.invoke("What is the AI in online?")
	print(response.content)
except Exception as e:
	print(f"Error: {e}")
	print("Consider upgrading your Gemini API plan or waiting for quota reset.")

The "AI in online" refers to the **application and integration of Artificial Intelligence technologies within the digital realm and online services.** It's a broad term that encompasses a wide range of ways AI is used to enhance, automate, personalize, and analyze online experiences.

Here's a breakdown of what "AI in online" generally means, categorized by its common applications:

**1. Enhancing User Experience & Personalization:**

*   **Recommendation Engines:** AI powers the personalized suggestions you see on platforms like Netflix, Amazon, Spotify, and YouTube. It analyzes your past behavior, preferences, and the behavior of similar users to recommend content, products, or services you're likely to enjoy.
*   **Personalized Content Delivery:** Websites and apps use AI to tailor the content they show you based on your interests, location, and past interactions. This can include news feeds, advertisements, and even website layouts.
*   **Chatbots and Virtual Assistants:** AI-power

In [4]:
# RAG IMPLEMENTATION with own data.

from langchain_core.documents import Document
# Sample documents for RAG
mytext = """India, officially the Republic of India,[j][20] is a country in South Asia. It is the seventh-largest country by area; \
          the most populous country since 2023;[21] and, since its independence in 1947, the world's most populous democracy.[22][23][24] \
          Bounded by the Indian Ocean on the south, the Arabian Sea on the southwest, and the Bay of Bengal on the southeast, it shares land\
               borders with Pakistan to the west;[k] China, Nepal, and Bhutan to the north; and Bangladesh and Myanmar to the east. In the \
Indian Ocean, India is near Sri Lanka and the Maldives; its Andaman and Nicobar Islands share a maritime border with Myanmar, Thailand, and Indonesia. \

 Modern humans arrived on the Indian subcontinent from Africa no later than 55,000 years ago.[26][27][28] Their long occupation, predominantly in isolation \
  as hunter-gatherers, has made the region highly diverse.[29] Settled life emerged on the subcontinent in the western margins of the Indus river basin 9,000 years ago, \
  evolving gradually into the Indus Valley Civilisation of the third millennium BCE.[30] By 1200 BCE, an archaic form of Sanskrit, an Indo-European language, had diffused into\
 India from the northwest.[31][32] Its hymns recorded the early dawnings of Hinduism in India.[33] India's pre-existing Dravidian languages were supplanted in the northern regions. \
 [34] By 400 BCE, caste had emerged within Hinduism,[35] and Buddhism and Jainism had arisen, proclaiming social orders unlinked to heredity.[36] Early political consolidations gave  \
 rise to the loose-knit Maurya and Gupta Empires.[37] This era was noted for creativity in art, architecture, and writing,[38] but the status of women declined,[39] and untouchability  \
 became an organised belief.[l][40] In South India, the Middle kingdoms exported Dravidian language scripts and religious cultures to the kingdoms of Southeast Asia.[41]"""

docs = [ Document(page_content=mytext, metadata={"source": "wikipedia.org", "title": "India - Wikipedia"})]

docs

[Document(metadata={'source': 'wikipedia.org', 'title': 'India - Wikipedia'}, page_content="India, officially the Republic of India,[j][20] is a country in South Asia. It is the seventh-largest country by area;           the most populous country since 2023;[21] and, since its independence in 1947, the world's most populous democracy.[22][23][24]           Bounded by the Indian Ocean on the south, the Arabian Sea on the southwest, and the Bay of Bengal on the southeast, it shares land               borders with Pakistan to the west;[k] China, Nepal, and Bhutan to the north; and Bangladesh and Myanmar to the east. In the Indian Ocean, India is near Sri Lanka and the Maldives; its Andaman and Nicobar Islands share a maritime border with Myanmar, Thailand, and Indonesia. \n Modern humans arrived on the Indian subcontinent from Africa no later than 55,000 years ago.[26][27][28] Their long occupation, predominantly in isolation   as hunter-gatherers, has made the region highly diverse.[29] 

In [5]:
# converting the doc into chunks 

from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Initialize the splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # Maximum size of each chunk
    chunk_overlap=50,     # Overlap to ensure context isn't lost at the edges
    length_function=len,   # How to measure the size (characters)
    add_start_index=True,  # Helps track where in the doc the chunk came from
)

# 2. Split the documents
chunks = text_splitter.split_documents(docs)

# 3. Verify the output
print(f"Original documents: {len(docs)}")
print(f"Created {len(chunks)} chunks.")

# Print the first chunk to see the content and metadata
print("\n--- First Chunk Content ---")
print(chunks[0].page_content)
print("\n--- Metadata (includes source and start_index) ---")
print(chunks[0].metadata)

Original documents: 1
Created 5 chunks.

--- First Chunk Content ---
India, officially the Republic of India,[j][20] is a country in South Asia. It is the seventh-largest country by area;           the most populous country since 2023;[21] and, since its independence in 1947, the world's most populous democracy.[22][23][24]           Bounded by the Indian Ocean on the south, the Arabian Sea on the southwest, and the Bay of Bengal on the southeast, it shares land               borders with Pakistan to the west;[k] China, Nepal, and Bhutan to the north; and

--- Metadata (includes source and start_index) ---
{'source': 'wikipedia.org', 'title': 'India - Wikipedia', 'start_index': 0}


In [6]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2-preview")

print("Generating embeddings with Gemini Embedding 2...")
vector_list = embeddings.embed_documents([chunk.page_content for chunk in chunks])

print(f"Success! Dimension size: {len(vector_list[0])}")

Generating embeddings with Gemini Embedding 2...
Success! Dimension size: 3072


In [7]:
from langchain_chroma import Chroma
persist_directory = "../chroma_db"

print("Saving embeddings to Chroma...")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings, 
    persist_directory=persist_directory
)

print(f"Success! Vector store saved to {persist_directory}")


Saving embeddings to Chroma...
Success! Vector store saved to ../chroma_db


In [8]:
vectorstore.similarity_search("What is the history of India?", k=2)

[Document(id='e7b5a4b3-97a8-4546-a5be-802d75ca866b', metadata={'start_index': 689, 'title': 'India - Wikipedia', 'source': 'wikipedia.org'}, page_content='Modern humans arrived on the Indian subcontinent from Africa no later than 55,000 years ago.[26][27][28] Their long occupation, predominantly in isolation   as hunter-gatherers, has made the region highly diverse.[29] Settled life emerged on the subcontinent in the western margins of the Indus river basin 9,000 years ago,   evolving gradually into the Indus Valley Civilisation of the third millennium BCE.[30] By 1200 BCE, an archaic form of Sanskrit, an Indo-European language, had diffused'),
 Document(id='09bac6f7-ba6c-448e-810a-0f023140c386', metadata={'title': 'India - Wikipedia', 'start_index': 689, 'source': 'wikipedia.org'}, page_content='Modern humans arrived on the Indian subcontinent from Africa no later than 55,000 years ago.[26][27][28] Their long occupation, predominantly in isolation   as hunter-gatherers, has made the r

In [9]:
# talk to llm

context = vectorstore.similarity_search("What is the history of India?", k=2)

response = llm.invoke(f"What is the history of India? you can answer using the following context: {context}")

print(response.content)

The history of India begins with the arrival of modern humans from Africa around 55,000 years ago. These early inhabitants lived as hunter-gatherers, contributing to the region's diversity. Settled life began to emerge approximately 9,000 years ago in the Indus River basin, eventually developing into the Indus Valley Civilisation by the third millennium BCE. By 1200 BCE, an early form of Sanskrit, an Indo-European language, had spread throughout the subcontinent.
